In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\davis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Accessing Youtube Transcript

In [ ]:
ytt = YouTubeTranscriptApi()
transcript_object = ytt.fetch(video_id= "bc3UYGFE8DI", languages=["de"])
print(transcript_object)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='wenn ihr nach deutschland kommt dann', start=0.0, duration=5.279), FetchedTranscriptSnippet(text='werdet ihr hier viele bäckereien sehen', start=2.31, duration=5.7), FetchedTranscriptSnippet(text='bäckereien sind sehr populär und viele', start=5.279, duration=5.15), FetchedTranscriptSnippet(text='deutsche kaufen da ihr frühstück', start=8.01, duration=5.31), FetchedTranscriptSnippet(text='mittagessen und auch nochmal ihr', start=10.429, duration=4.781), FetchedTranscriptSnippet(text='nachmittags kuchen ein', start=13.32, duration=4.32), FetchedTranscriptSnippet(text='und weil ihr diese orte bestimmt oft', start=15.21, duration=5.31), FetchedTranscriptSnippet(text='besucht zeigen wir euch heute wie man in', start=17.64, duration=5.479), FetchedTranscriptSnippet(text='einer deutschen bäckerei etwas bestellt', start=20.52, duration=4.55), FetchedTranscriptSnippet(text="los geht's", start=23.119, duration=12.691), FetchedTranscript

In [ ]:
subtitles = []
for snippet in transcript_object:
    print(snippet.text)
    subtitles.append(snippet.text)

wenn ihr nach deutschland kommt dann
werdet ihr hier viele bäckereien sehen
bäckereien sind sehr populär und viele
deutsche kaufen da ihr frühstück
mittagessen und auch nochmal ihr
nachmittags kuchen ein
und weil ihr diese orte bestimmt oft
besucht zeigen wir euch heute wie man in
einer deutschen bäckerei etwas bestellt
los geht's
[Musik]
zwei normale brötchen bitte ich hätte
gern einmal bitte zwei normale brötchen
ja zwei normale brötchen hätte ich gern
ich hätte gerne zwei normale brötchen
und drei mehr korn brötchen dazu nehme
ich auch noch drei mehr kongolesen bitte
und dann kommen noch drei mehr können
brötchen dazu zwei mehr kommen brötchen
und drei kürbis kein brötchen zwei
croissants bitte und jetzt zwei crosser
und noch dazu zwei croissants bitter ich
hätte gerne zwei croissants eine brezel
bitte guten tag ich hätte gerne eine
brezel bitte noch eine brezel wird kann
ich bitte eine brezel haben ein baguette
bitte ich hätte gern ein baguette bitte
ein wenig beeilen
belegte brötc

In [ ]:
print(subtitles)

['wenn ihr nach deutschland kommt dann', 'werdet ihr hier viele bäckereien sehen', 'bäckereien sind sehr populär und viele', 'deutsche kaufen da ihr frühstück', 'mittagessen und auch nochmal ihr', 'nachmittags kuchen ein', 'und weil ihr diese orte bestimmt oft', 'besucht zeigen wir euch heute wie man in', 'einer deutschen bäckerei etwas bestellt', "los geht's", '[Musik]', 'zwei normale brötchen bitte ich hätte', 'gern einmal bitte zwei normale brötchen', 'ja zwei normale brötchen hätte ich gern', 'ich hätte gerne zwei normale brötchen', 'und drei mehr korn brötchen dazu nehme', 'ich auch noch drei mehr kongolesen bitte', 'und dann kommen noch drei mehr können', 'brötchen dazu zwei mehr kommen brötchen', 'und drei kürbis kein brötchen zwei', 'croissants bitte und jetzt zwei crosser', 'und noch dazu zwei croissants bitter ich', 'hätte gerne zwei croissants eine brezel', 'bitte guten tag ich hätte gerne eine', 'brezel bitte noch eine brezel wird kann', 'ich bitte eine brezel haben ein bag

In [ ]:
final_transcript = " ".join(subtitles)
print(final_transcript)

wenn ihr nach deutschland kommt dann werdet ihr hier viele bäckereien sehen bäckereien sind sehr populär und viele deutsche kaufen da ihr frühstück mittagessen und auch nochmal ihr nachmittags kuchen ein und weil ihr diese orte bestimmt oft besucht zeigen wir euch heute wie man in einer deutschen bäckerei etwas bestellt los geht's [Musik] zwei normale brötchen bitte ich hätte gern einmal bitte zwei normale brötchen ja zwei normale brötchen hätte ich gern ich hätte gerne zwei normale brötchen und drei mehr korn brötchen dazu nehme ich auch noch drei mehr kongolesen bitte und dann kommen noch drei mehr können brötchen dazu zwei mehr kommen brötchen und drei kürbis kein brötchen zwei croissants bitte und jetzt zwei crosser und noch dazu zwei croissants bitter ich hätte gerne zwei croissants eine brezel bitte guten tag ich hätte gerne eine brezel bitte noch eine brezel wird kann ich bitte eine brezel haben ein baguette bitte ich hätte gern ein baguette bitte ein wenig beeilen belegte brötc

Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 20)
list_of_docs = splitter.create_documents(texts=[final_transcript])

In [ ]:
print(list_of_docs)

[Document(metadata={}, page_content='wenn ihr nach deutschland kommt dann werdet ihr hier viele bäckereien sehen bäckereien sind sehr populär und viele deutsche kaufen da ihr frühstück mittagessen und auch nochmal ihr nachmittags kuchen'), Document(metadata={}, page_content="nachmittags kuchen ein und weil ihr diese orte bestimmt oft besucht zeigen wir euch heute wie man in einer deutschen bäckerei etwas bestellt los geht's [Musik] zwei normale brötchen bitte ich hätte"), Document(metadata={}, page_content='bitte ich hätte gern einmal bitte zwei normale brötchen ja zwei normale brötchen hätte ich gern ich hätte gerne zwei normale brötchen und drei mehr korn brötchen dazu nehme ich auch noch drei mehr'), Document(metadata={}, page_content='auch noch drei mehr kongolesen bitte und dann kommen noch drei mehr können brötchen dazu zwei mehr kommen brötchen und drei kürbis kein brötchen zwei croissants bitte und jetzt zwei crosser und noch'), Document(metadata={}, page_content='crosser und n

In [ ]:
print(len(list_of_docs))

15


Vector Store (FAISS)

In [ ]:
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [ ]:
vector_store =FAISS.from_documents(documents=list_of_docs, embedding=embeddings)

Retriever

In [ ]:
retriever = vector_store.as_retriever()

In [ ]:
question = "What are the different types of bread available in a German bakery?"

Langchain Pipleline

In [ ]:
llm = ChatOpenAI(model = 'gpt-3.5-turbo')
template = PromptTemplate(template = "Translate the following question into German {topic}",
                            input_variables = ['topic'])
parser = StrOutputParser()

In [ ]:
chain = template | llm | parser
result = chain.invoke({'topic':question})

In [ ]:
result

'Welche verschiedenen Arten von Brot gibt es in einer deutschen Bäckerei?'

RAG Retrieval

In [ ]:
RAG_result = retriever.invoke(question)

In [ ]:
print(len(RAG_result))

4


In [ ]:
print(RAG_result)

[Document(id='74a0c92f-7bca-4cf0-883f-a087709c3dae', metadata={}, page_content='wenn ihr nach deutschland kommt dann werdet ihr hier viele bäckereien sehen bäckereien sind sehr populär und viele deutsche kaufen da ihr frühstück mittagessen und auch nochmal ihr nachmittags kuchen'), Document(id='e65b33ea-0e91-4460-bda1-1d74a5272292', metadata={}, page_content="nachmittags kuchen ein und weil ihr diese orte bestimmt oft besucht zeigen wir euch heute wie man in einer deutschen bäckerei etwas bestellt los geht's [Musik] zwei normale brötchen bitte ich hätte"), Document(id='49ac3861-d43b-4401-b3f5-aa8507f15e76', metadata={}, page_content='bestellen wir mal tatsächlich etwas in einer bäckerei ja nicht was hättest du gerne ich hätte gerne zwei mehr kommen wir hätten gerne ein stück käsekuchen bitte außerdem hätten wir gerne noch einen'), Document(id='4b0a8195-b20b-420a-9582-5e88344b647e', metadata={}, page_content='gerne ein belegtes brötchen mit schinken ein belegtes brötchen mit mett und zw

In [ ]:
for i, res in enumerate(RAG_result):
    print("Result:",i+1)
    print(res.page_content)

Result: 1
wenn ihr nach deutschland kommt dann werdet ihr hier viele bäckereien sehen bäckereien sind sehr populär und viele deutsche kaufen da ihr frühstück mittagessen und auch nochmal ihr nachmittags kuchen
Result: 2
nachmittags kuchen ein und weil ihr diese orte bestimmt oft besucht zeigen wir euch heute wie man in einer deutschen bäckerei etwas bestellt los geht's [Musik] zwei normale brötchen bitte ich hätte
Result: 3
bestellen wir mal tatsächlich etwas in einer bäckerei ja nicht was hättest du gerne ich hätte gerne zwei mehr kommen wir hätten gerne ein stück käsekuchen bitte außerdem hätten wir gerne noch einen
Result: 4
gerne ein belegtes brötchen mit schinken ein belegtes brötchen mit mett und zwiebeln ich hätte gerne ein brot ich hätte gern ein brot hallo ich hätte gern ein brot bitte ich hätte gern ein brot


In [ ]:
doc_list = []
for res in RAG_result:
    doc_list.append(res.page_content)

In [ ]:
print(doc_list)

['wenn ihr nach deutschland kommt dann werdet ihr hier viele bäckereien sehen bäckereien sind sehr populär und viele deutsche kaufen da ihr frühstück mittagessen und auch nochmal ihr nachmittags kuchen', "nachmittags kuchen ein und weil ihr diese orte bestimmt oft besucht zeigen wir euch heute wie man in einer deutschen bäckerei etwas bestellt los geht's [Musik] zwei normale brötchen bitte ich hätte", 'bestellen wir mal tatsächlich etwas in einer bäckerei ja nicht was hättest du gerne ich hätte gerne zwei mehr kommen wir hätten gerne ein stück käsekuchen bitte außerdem hätten wir gerne noch einen', 'gerne ein belegtes brötchen mit schinken ein belegtes brötchen mit mett und zwiebeln ich hätte gerne ein brot ich hätte gern ein brot hallo ich hätte gern ein brot bitte ich hätte gern ein brot']


In [ ]:
augmented_text = "\n\n".join(doc_list)
print(augmented_text)

wenn ihr nach deutschland kommt dann werdet ihr hier viele bäckereien sehen bäckereien sind sehr populär und viele deutsche kaufen da ihr frühstück mittagessen und auch nochmal ihr nachmittags kuchen

nachmittags kuchen ein und weil ihr diese orte bestimmt oft besucht zeigen wir euch heute wie man in einer deutschen bäckerei etwas bestellt los geht's [Musik] zwei normale brötchen bitte ich hätte

bestellen wir mal tatsächlich etwas in einer bäckerei ja nicht was hättest du gerne ich hätte gerne zwei mehr kommen wir hätten gerne ein stück käsekuchen bitte außerdem hätten wir gerne noch einen

gerne ein belegtes brötchen mit schinken ein belegtes brötchen mit mett und zwiebeln ich hätte gerne ein brot ich hätte gern ein brot hallo ich hätte gern ein brot bitte ich hätte gern ein brot


Langchain pipeline

In [ ]:
template2 = PromptTemplate(template = """You are a helpful German assistant who answers the question {question}
                           from the provided context {context}. 
                           If the context is insufficient, just say it 
                           is not mentioned in the given context. """,
                            input_variables = ['question','context'])

In [ ]:
chain2 = RunnableSequence(template2, llm, parser)
final_result = chain2.invoke({'question': result, 'context': augmented_text})

In [ ]:
print(final_result)

In einer deutschen Bäckerei gibt es verschiedene Arten von Brot, wie zum Beispiel normale Brötchen, belegte Brötchen mit Schinken, belegte Brötchen mit Mett und Zwiebeln, Käsekuchen und verschiedenen Brotsorten.
